# Project overview

This notebook is the canonical exploratory notebook for the project. It includes a concise overview of repository files, outputs, and quick run commands for the three tasks (easy, medium, hard).

**Quick smoke-run commands**
- Easy: `SAMPLE_SIZE=20 EPOCHS=1 python run_easy_task.py`
- Medium: `SAMPLE_SIZE=20 EPOCHS=1 python run_medium_task.py` (no genre usage by default; set `N_CLUSTERS` if needed)
- Hard: `RESOLVE_PATHS_ONLY=1 python run_hard_task.py` (checks file paths; set `SAMPLE_SIZE`/`EPOCHS` for a short run)

**Key files**
- `run_easy_task.py` — VAE on mel-spectrograms; t-SNE on latent; PCA + KMeans baseline
- `run_medium_task.py` — ConvVAE (audio + lyrics); KMeans/Agglomerative/DBSCAN on latent; PCA (hybrid) baseline
- `run_hard_task.py` — Beta-VAE with optional genre conditioning; saves reconstructions and baselines
- `src/` — dataset loaders, models, clustering helpers, evaluation and visualization utilities
- `results/` — metrics CSVs, indices CSVs, `latent_visualization/`, `reconstructions/hard/`, and `clusters_*.csv`

**Notes**
- Visuals default to **t-SNE** with adaptive perplexity.
- Medium task does **not** use genre labels for clustering by default.
- For full details see `README.md`.


# Exploratory Analysis: Unsupervised Clustering of Hybrid-Language Music
This notebook demonstrates data loading, feature extraction, training a simple Variational Autoencoder (VAE), clustering, and visualization for the hybrid-language music clustering project.

## Requirements
- librosa
- sentence-transformers
- torch
- sklearn
- umap-learn
- matplotlib

Make sure to install the required packages before running the notebook.
Run this in your terminal:
`pip install librosa sentence-transformers torch sklearn umap-learn matplotlib`

In [ ]:
# Quick utilities: preview README and list top-level files
import pathlib, os, glob
p = pathlib.Path('README.md')
print('README preview:\n')
if p.exists():
    print(p.read_text()[:1200])
else:
    print('README.md not found in repo root')

print('\nTop-level files:')
for f in sorted(os.listdir('.')):
    print(f)

print('\nResults overview:')
for f in sorted(glob.glob('results/*'))[:20]:
    print(f)

## 1. Import libraries

In [ ]:

import os
import numpy as np
import librosa
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans, GaussianMixture
from sklearn.preprocessing import StandardScaler
import umap
import matplotlib.pyplot as plt


## 2. Dataset Loading (Placeholder)
Replace this section with your actual dataset loading code. Here we assume you have:
- Audio files in a folder
- Corresponding lyrics in text files
- A mapping of file names to labels if available

In [ ]:

# Example dummy dataset class
class MusicDataset(Dataset):
    def __init__(self, audio_dir, lyrics_dir, sr=22050, duration=5):
        self.audio_dir = audio_dir
        self.lyrics_dir = lyrics_dir
        self.sr = sr
        self.duration = duration  # seconds
        self.file_list = [f[:-4] for f in os.listdir(audio_dir) if f.endswith('.wav')]
        self.lyrics_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        fname = self.file_list[idx]
        
        # Load audio and convert to mel-spectrogram
        audio_path = os.path.join(self.audio_dir, fname + '.wav')
        y, _ = librosa.load(audio_path, sr=self.sr, duration=self.duration)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=self.sr, n_mels=64)
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_db = mel_db.astype(np.float32)
        
        # Load and embed lyrics
        lyrics_path = os.path.join(self.lyrics_dir, fname + '.txt')
        with open(lyrics_path, 'r', encoding='utf-8') as f:
            lyrics = f.read()
        lyrics_emb = self.lyrics_model.encode(lyrics)
        lyrics_emb = lyrics_emb.astype(np.float32)
        
        return mel_db, lyrics_emb


## 3. Variational Autoencoder (VAE) Model
Basic VAE architecture to encode audio and lyrics features into a latent space.

In [ ]:

class VAE(nn.Module):
    def __init__(self, audio_dim=(64, 216), lyrics_dim=384, latent_dim=32):
        super(VAE, self).__init__()
        
        self.audio_dim = audio_dim
        self.lyrics_dim = lyrics_dim
        self.latent_dim = latent_dim
        
        # Flatten audio input size
        self.audio_flat_dim = audio_dim[0] * audio_dim[1]
        
        # Encoder
        self.fc1 = nn.Linear(self.audio_flat_dim + lyrics_dim, 512)
        self.fc2_mu = nn.Linear(512, latent_dim)
        self.fc2_logvar = nn.Linear(512, latent_dim)
        
        # Decoder
        self.fc3 = nn.Linear(latent_dim, 512)
        self.fc4 = nn.Linear(512, self.audio_flat_dim + lyrics_dim)
        
    def encode(self, x_audio, x_lyrics):
        x_audio = x_audio.view(-1, self.audio_flat_dim)
        x = torch.cat([x_audio, x_lyrics], dim=1)
        h1 = torch.relu(self.fc1(x))
        return self.fc2_mu(h1), self.fc2_logvar(h1)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h3 = torch.relu(self.fc3(z))
        recon = self.fc4(h3)
        return recon
    
    def forward(self, x_audio, x_lyrics):
        mu, logvar = self.encode(x_audio, x_lyrics)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


## 4. Loss Function and Training Loop

In [ ]:

def loss_function(recon_x, x, mu, logvar):
    BCE = nn.functional.mse_loss(recon_x, x, reduction='sum')
    # KL divergence term
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

def train_vae(model, dataloader, epochs=20, lr=1e-3, device='cpu'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        train_loss = 0
        for mel_db, lyrics_emb in dataloader:
            mel_db = mel_db.to(device)
            lyrics_emb = lyrics_emb.to(device)
            
            optimizer.zero_grad()
            # Flatten inputs
            mel_db_flat = mel_db.view(mel_db.size(0), -1)
            lyrics_emb_flat = lyrics_emb
            inputs = torch.cat([mel_db_flat, lyrics_emb_flat], dim=1)
            
            recon, mu, logvar = model(mel_db, lyrics_emb)
            loss = loss_function(recon, inputs, mu, logvar)
            loss.backward()
            train_loss += loss.item()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {train_loss / len(dataloader.dataset):.4f}")
    return model


## 5. Extract Latent Representations

In [ ]:

def get_latent_representations(model, dataloader, device='cpu'):
    model.eval()
    latents = []
    with torch.no_grad():
        for mel_db, lyrics_emb in dataloader:
            mel_db = mel_db.to(device)
            lyrics_emb = lyrics_emb.to(device)
            mu, _ = model.encode(mel_db, lyrics_emb)
            latents.append(mu.cpu().numpy())
    return np.vstack(latents)


## 6. Clustering Latent Space

In [ ]:

def cluster_latent_space(latent_vectors, n_clusters=10):
    scaler = StandardScaler()
    latent_scaled = scaler.fit_transform(latent_vectors)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    gmm = GaussianMixture(n_components=n_clusters, random_state=42)
    
    kmeans_labels = kmeans.fit_predict(latent_scaled)
    gmm_labels = gmm.fit_predict(latent_scaled)
    
    return kmeans_labels, gmm_labels


## 7. Visualize Clusters with UMAP

In [ ]:

def visualize_umap(latent_vectors, labels=None, title='UMAP projection'):
    reducer = umap.UMAP(random_state=42)
    embedding = reducer.fit_transform(latent_vectors)
    
    plt.figure(figsize=(8,6))
    if labels is not None:
        scatter = plt.scatter(embedding[:,0], embedding[:,1], c=labels, cmap='Spectral', s=10)
        plt.legend(*scatter.legend_elements(), title="Clusters")
    else:
        plt.scatter(embedding[:,0], embedding[:,1], s=10)
    plt.title(title)
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.show()


## 8. Usage Example (Fill in your data paths)

In [ ]:

# Replace these paths with your dataset locations
audio_dir = '/path/to/audio_files'
lyrics_dir = '/path/to/lyrics_files'

dataset = MusicDataset(audio_dir, lyrics_dir)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Initialize and train VAE
vae_model = VAE()
trained_vae = train_vae(vae_model, dataloader, epochs=10, device=device)

# Extract latent vectors
latent_vectors = get_latent_representations(trained_vae, dataloader, device=device)

# Cluster latent space
kmeans_labels, gmm_labels = cluster_latent_space(latent_vectors, n_clusters=10)

# Visualize clusters
visualize_umap(latent_vectors, kmeans_labels, title='UMAP of VAE Latent Space with K-Means Clusters')
